# System Contribution to Delivered Image Quality — Design-Spec Verification

**Question.** Do on-sky LSSTCam data contradict the image-quality design specification below?

> **Discussion:** The design specification for image quality requires that, for median atmospheric
> seeing, the system contribution to the delivered image quality never exceed 15%. This requirement
> should be fulfilled irrespective of the airmass, which limits the seeing degradation due to
> hardware away from the zenith (e.g., due to gravity load). Assuming that the atmospheric seeing
> increases with airmass, $X$, as $X^{0.6}$, the design specification for the allowed error budget
> due to the system is **0.52 arcsec at an airmass of 2** and for the median seeing conditions
> (**0.42 arcsec for $X=1.4$**).

**Measurement.** The system term is ConsDB `cdb_lsstcam.visit1_quicklook.aos_fwhm` — *"estimated
contribution of the optical system to the overall seeing (FWHM) using the measured zernike
coefficients"* (arcsec), combined in quadrature with the camera/detector floor
`CAM_FWHM = 0.207″` used by `rubin_nights` to form the delivered instrumental term
`idiq_aos_cam = √(aos_fwhm² + CAM_FWHM²)`.

## What this notebook is, and is not

Following the convention established in
[Wind_Loading_and_Ingression_Operational_Limits.ipynb](../dome_and_wind/Wind_Loading_and_Ingression_Operational_Limits.ipynb),
this is a **contradiction test**, not a compliance certification. The difference from the wind
notebook is important and runs the other way:

| | wind notebook | **this notebook** |
|---|---|---|
| is the design point reached on sky? | **no** (max 18.9 of 20 m/s) → extrapolation | **yes** — every exposure has a measured $X$ and a measured `aos_fwhm` |
| dominant limitation | statistical power | **estimator validity** (is `aos_fwhm` clean?) |
| available verdicts | mostly UNDERPOWERED | a real **CONTRADICTED** is reachable |

Because the spec is evaluated *at the airmass of each exposure*, no extrapolation is needed. The
burden therefore shifts entirely onto the guards in §4 — the risk here is not "too few data" but
"`aos_fwhm` contaminated by conditions the spec did not intend".

## Structure

- **§2** decode the spec — the two quoted numbers over-determine it, which lets us *verify* our reading
- **§3** load per-exposure ConsDB data (256 nights)
- **§4** guards: median-seeing, DIMM-validity wind floor, open dome
- **§5** sample characterisation + selection-bias diagnostic
- **§6** primary verdict $U = \mathrm{system}/\mathrm{budget}$
- **§7** the airmass clause (gravity load) — within-night, placebo, positive control
- **§8** guard sensitivity — does the verdict depend on our cuts?
- **§9** time trend — commissioning maturation
- **§10** budget composition — where the budget actually goes
- **§11** verdict table

## 1. Setup

In [ ]:
import os
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlalchemy
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# ── ConsDB / PostgreSQL (idiom shared with IQ_vs_SunElevation.ipynb) ──────────
PGPASS_FILE = os.path.expanduser("~/.lsst/postgres-credentials.txt")
CONSDB_HOST = "usdf-summitdb-logical-replica-svc.sdf.slac.stanford.edu"
CONSDB_DB = "exposurelog"
CONSDB_USER = "usdf"
SCHEMA = "cdb_lsstcam"


def load_pgpass(path, host, database, user):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split(":")
            if len(parts) < 5:
                continue
            if parts[0] == host and parts[2] == database and parts[3] == user:
                return ":".join(parts[4:])
    raise ValueError(f"No credentials for {user}@{host}/{database}")


engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{CONSDB_USER}:"
    f"{load_pgpass(PGPASS_FILE, CONSDB_HOST, CONSDB_DB, CONSDB_USER)}@{CONSDB_HOST}/{CONSDB_DB}",
    connect_args={"connect_timeout": 60},
)


def consdb_query(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(sqlalchemy.text(sql), conn)


CACHE_DIR = pathlib.Path("../data")
CACHE = CACHE_DIR / "aos_system_iq_spec.parquet"

# ── analysis constants ───────────────────────────────────────────────────────
PIXEL_SCALE = 0.2  # arcsec/pix (LSSTCam)
SIGMA_TO_FWHM = 2.355
CAM_FWHM = 0.207  # arcsec — camera/detector floor (rubin_nights.rubin_scheduler_addons)
ALPHA = 0.6  # seeing airmass exponent, theta ~ X^0.6 (spec's own assumption)

# DIMM has a bad-value tail out to ~40"; Cerro Pachon seeing is 0.4-2.5".
DIMM_VALID = (0.2, 3.0)
RNG = np.random.default_rng(20260817)
print("setup ok")

## 2. Decoding the specification

The spec text quotes **two** numbers (0.52″ at $X=2$, 0.42″ at $X=1.4$) for what is a *one*-parameter
family. That redundancy is a gift: it lets us test our interpretation rather than assume it.

**Step 1 — is the pair consistent with the stated $X^{0.6}$ scaling?**

$$\frac{0.52}{0.42} = 1.2381 \qquad\text{vs}\qquad \left(\frac{2}{1.4}\right)^{0.6} = 1.2386$$

Agreement to 0.04%. So the budget itself scales as $X^{0.6}$ — the system allowance *grows* with
airmass because the atmosphere it is being compared against does.

**Step 2 — what does "never exceed 15%" mean?** Two readings are possible; only one reproduces the
quoted numbers:

| reading | system/atmosphere | implied zenith seeing $\theta_0$ |
|---|---|---|
| **quadrature**: delivered $=1.15\times$ atmospheric, i.e. $\mathrm{sys}=\theta_{\rm atm}\sqrt{1.15^2-1}$ | 0.5679 | **0.6041″** (from $X{=}2$), **0.6044″** (from $X{=}1.4$) |
| variance: $\mathrm{sys}^2 = 0.15\,\theta_{\rm atm}^2$ | 0.3873 | 0.886″ — not a recognisable median seeing |

The quadrature reading returns **the same $\theta_0$ from both quoted numbers independently**, and
that value is $0.604''$ — a sensible Cerro Pachón median. This is a genuine consistency check, and it
passes.

**Step 3 — reconciling with the 0.69″ median.** The median seeing quoted for the site
*uncorrected for outer scale* is **0.69″**. Note that

$$\frac{0.69}{0.604} = 1.142$$

i.e. the spec's implicit base is the **outer-scale-corrected** median seeing, while 0.69″ is the raw
(von Kármán → Kolmogorov equivalent) figure. Both refer to the same conditions. This matters twice:

1. It confirms the decode — 0.604″ is not an arbitrary fit, it is the corrected form of a known number.
2. It sets the **seeing guard** in §4: "for median atmospheric seeing" is a *condition of the spec*,
   and the quantity we can cut on (`dimm_seeing`) is a raw, uncorrected measurement, so the guard
   threshold is the raw **0.69″**, not 0.604″.

**Adopted budget line** (strict, airmass-scaled — the reading the spec's own two points imply):

$$\boxed{\ \mathrm{budget}(X) = 0.52 \left(\frac{X}{2}\right)^{0.6} \mathrm{arcsec}\ }$$

which is $0.343''$ at zenith, $0.420''$ at $X{=}1.4$, $0.520''$ at $X{=}2$. The clause *"should be
fulfilled irrespective of the airmass"* is what forces us to apply it at **every** $X$, including
zenith where it is tightest — a flat $0.52''$ ceiling would render the airmass clause untestable.

In [ ]:
# ── spec decode, verified rather than assumed ────────────────────────────────
SPEC_PCT = 0.15
SPEC_POINTS = {2.0: 0.52, 1.4: 0.42}  # airmass -> allowed system FWHM [arcsec]

# quadrature reading: delivered = (1+pct) * atmospheric  =>  sys = atm * sqrt((1+pct)^2 - 1)
FRAC_QUAD = np.sqrt((1 + SPEC_PCT) ** 2 - 1)
FRAC_VAR = np.sqrt(SPEC_PCT)  # the alternative (variance) reading

print("Step 1 — is the quoted pair consistent with X^0.6?")
r_quoted = SPEC_POINTS[2.0] / SPEC_POINTS[1.4]
r_scale = (2.0 / 1.4) ** ALPHA
print(f"  0.52/0.42          = {r_quoted:.6f}")
print(f"  (2/1.4)^0.6        = {r_scale:.6f}")
print(
    f"  relative agreement = {abs(r_quoted / r_scale - 1) * 100:.3f}%  -> CONSISTENT\n"
)

print("Step 2 — which reading of '15%' reproduces both numbers?")
for name, frac in [
    ("quadrature (1.15x delivered)", FRAC_QUAD),
    ("variance (0.15*atm^2)", FRAC_VAR),
]:
    th = {X: b / (frac * X**ALPHA) for X, b in SPEC_POINTS.items()}
    vals = list(th.values())
    print(
        f"  {name:30s} sys/atm={frac:.4f}  theta0: "
        + ", ".join(f"X={X}->{v:.4f}" for X, v in th.items())
        + f"   spread={abs(vals[0] - vals[1]) * 1000:.2f} mas"
    )

THETA0_CORR = SPEC_POINTS[2.0] / (FRAC_QUAD * 2.0**ALPHA)
SEEING_MED_RAW = 0.69  # uncorrected-for-outer-scale median seeing (site figure)
print(
    f"\n  -> adopted: quadrature, implied corrected zenith seeing theta0 = {THETA0_CORR:.4f} arcsec"
)

print("\nStep 3 — reconcile with the raw (uncorrected) median seeing 0.69 arcsec")
print(
    f"  0.69 / {THETA0_CORR:.4f} = {SEEING_MED_RAW / THETA0_CORR:.4f}   (outer-scale correction factor)"
)
print(
    "  => spec base is the OUTER-SCALE-CORRECTED median; the DIMM guard must use the RAW 0.69."
)


def budget(X):
    """Allowed system FWHM [arcsec] at airmass X (strict, airmass-scaled)."""
    return SPEC_POINTS[2.0] * (np.asarray(X, dtype=float) / 2.0) ** ALPHA


print("\nAdopted budget line  budget(X) = 0.52*(X/2)^0.6 :")
for X in (1.0, 1.2, 1.4, 1.6, 2.0, 2.5):
    print(
        f"  X={X:4.1f}  ->  {budget(X):.4f} arcsec"
        + ("   <- quoted" if X in SPEC_POINTS else "")
    )

# round-trip: the budget line must reproduce the quoted points exactly
assert np.allclose(
    [budget(X) for X in SPEC_POINTS], list(SPEC_POINTS.values()), atol=2e-3
)
print("\nround-trip against both quoted points: OK")

## 3. Data

Per-exposure join of `cdb_lsstcam.exposure` (airmass, DIMM, wind, vignetting) with
`visit1_quicklook` (`aos_fwhm`, `donut_blur_fwhm`, `psf_sigma_median`).

We query ConsDB directly rather than reusing
`data/wind_loading_20260107_20260714.parquet`, because that cache is built for the wind analysis and
covers 112 nights from 2026-01-07. This is an *optics* question with no wind-telemetry requirement,
so the full **256-night** ConsDB record from 2025-04-15 is available — more than twice the baseline,
which is what makes the maturation trend in §9 measurable.

In [ ]:
QUERY = f"""
SELECT e.exposure_id, e.day_obs, e.seq_num, e.obs_start, e.airmass, e.band, e.exp_time,
       e.dimm_seeing, e.wind_speed, e.wind_dir, e.air_temp, e.focus_z, e.sky_rotation,
       e.vignette, e.can_see_sky, e.scheduler_note,
       e.zenith_distance_start, e.zenith_distance_end,
       q.aos_fwhm, q.donut_blur_fwhm, q.psf_sigma_median, q.psf_area_median,
       q.eff_time_median, q.psf_ixx_median, q.psf_iyy_median, q.psf_ixy_median
FROM {SCHEMA}.exposure e
JOIN {SCHEMA}.visit1_quicklook q ON e.exposure_id = q.visit_id
WHERE e.img_type = 'science'
"""

if CACHE.exists():
    raw = pd.read_parquet(CACHE)
    print(f"loaded cache {CACHE.name}")
else:
    raw = consdb_query(QUERY)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    raw.to_parquet(CACHE)
    print(f"queried ConsDB and wrote cache {CACHE.name}")

NUMERIC = [
    "airmass",
    "aos_fwhm",
    "donut_blur_fwhm",
    "psf_sigma_median",
    "psf_area_median",
    "dimm_seeing",
    "wind_speed",
    "wind_dir",
    "exp_time",
    "air_temp",
    "eff_time_median",
]
for c in NUMERIC:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

print(f"\nscience exposures : {len(raw):,}")
print(
    f"nights            : {raw.day_obs.nunique()}  ({raw.day_obs.min()} -> {raw.day_obs.max()})"
)
print(
    f"aos_fwhm non-null : {raw.aos_fwhm.notna().sum():,} "
    f"({100 * raw.aos_fwhm.notna().mean():.1f}%)  on {raw.loc[raw.aos_fwhm.notna(), 'day_obs'].nunique()} nights"
)
print("\ncolumn coverage:")
for c in [
    "aos_fwhm",
    "donut_blur_fwhm",
    "psf_sigma_median",
    "dimm_seeing",
    "wind_speed",
    "airmass",
]:
    print(f"  {c:24s} {raw[c].notna().sum():7,d}")

## 4. Guards

The spec is conditional — *"for median atmospheric seeing"* — so a valid test cannot simply average
over all conditions. Four guards, each with a stated reason:

1. **`aos_fwhm` validity** `0.05–3.0″`. The raw column runs to 3.9″; values several times the
   atmospheric seeing are Zernike-fit failures, not optics.
2. **Median-seeing guard** `dimm_seeing ≤ 0.69″`. This is the spec's own condition. It also protects
   against the *masking* failure mode: in poor seeing the donut images that feed the Zernike
   estimate are themselves degraded, so `aos_fwhm` drifts upward with seeing (shown below). Cutting
   to at-or-better-than median seeing removes that contamination and makes the test **conservative** —
   we evaluate the optics in exactly the conditions where a real fault is *hardest* to hide behind
   the atmosphere.
3. **DIMM-reliability wind floor** `wind_speed ≥ 4 m/s`. In near-calm conditions the DIMM is
   unreliable — weak turbulence mixing and local/dome-seeing contamination at the DIMM tower — so
   low-wind seeing values cannot be trusted to establish "median seeing".
4. **Open dome / unvignetted** `can_see_sky` and not vignetted, matching the wind notebook's guard.

Guards 2+3 are strict and cost ~92% of exposures. §5 checks what survives is still representative
and §8 re-runs the verdict without each guard.

In [ ]:
# ── masking diagnostic: does aos_fwhm itself degrade in poor seeing? ─────────
chk = raw[raw.aos_fwhm.between(0.05, 3.0) & raw.airmass.between(1.0, 3.5)].copy()
chk["dimm"] = chk.dimm_seeing.where(chk.dimm_seeing.between(*DIMM_VALID))
chk = chk.dropna(subset=["dimm"])
chk["dimm_q"] = pd.qcut(chk.dimm, 5)
tbl = chk.groupby("dimm_q", observed=True).agg(
    n=("aos_fwhm", "size"),
    dimm_med=("dimm", "median"),
    aos_med=("aos_fwhm", "median"),
    aos_p90=("aos_fwhm", lambda s: s.quantile(0.90)),
)
print("aos_fwhm vs DIMM quintile — the masking/contamination check")
print(tbl.round(4).to_string())

r_pool = chk[["aos_fwhm", "dimm"]].corr().iloc[0, 1]
tmp = chk.copy()
for c in ["aos_fwhm", "dimm"]:
    tmp[c + "_w"] = tmp[c] - tmp.groupby("day_obs")[c].transform("median")
r_within = tmp[["aos_fwhm_w", "dimm_w"]].corr().iloc[0, 1]
print(f"\ncorr(aos_fwhm, DIMM): pooled {r_pool:+.4f}   within-night {r_within:+.4f}")
print(
    f"aos_fwhm median rises {tbl.aos_med.iloc[0]:.4f} -> {tbl.aos_med.iloc[-1]:.4f} arcsec "
    f"across the seeing range ({100 * (tbl.aos_med.iloc[-1] / tbl.aos_med.iloc[0] - 1):+.1f}%)"
)
print(
    "=> real but modest contamination; the seeing guard removes it AND is required by the spec."
)

In [ ]:
# ── apply guards, logging attrition ─────────────────────────────────────────
steps = []
d = raw.copy()
steps.append(("science exposures with quicklook", len(d), d.day_obs.nunique()))

d = d[d.aos_fwhm.between(0.05, 3.0)]
steps.append(("aos_fwhm valid (0.05-3.0 arcsec)", len(d), d.day_obs.nunique()))

d = d[d.airmass.between(1.0, 3.5)]
steps.append(("airmass 1.0-3.5", len(d), d.day_obs.nunique()))

d["dimm"] = d.dimm_seeing.where(d.dimm_seeing.between(*DIMM_VALID))
d = d.dropna(subset=["dimm"])
steps.append((f"DIMM valid {DIMM_VALID} arcsec", len(d), d.day_obs.nunique()))

open_dome = d.can_see_sky.fillna(True).astype(bool) & (
    ~d.vignette.astype(str).str.upper().isin(["FULLY", "PARTIALLY"])
)
d = d[open_dome]
steps.append(("open dome / unvignetted", len(d), d.day_obs.nunique()))

# `unguarded` keeps every conditioning cut but drops the two SPEC-CONDITION guards,
# so §8 can isolate their effect.
unguarded = d.copy()

d = d[d.dimm <= SEEING_MED_RAW]
steps.append(
    (f"median-seeing guard DIMM <= {SEEING_MED_RAW}", len(d), d.day_obs.nunique())
)

WIND_FLOOR = 4.0  # m/s — below this the DIMM is unreliable
d = d[d.wind_speed >= WIND_FLOOR]
steps.append(
    (f"DIMM-reliability wind floor >= {WIND_FLOOR} m/s", len(d), d.day_obs.nunique())
)

print(f"{'guard':44s} {'exposures':>10s} {'nights':>8s}")
print("-" * 64)
for name, n, nn in steps:
    print(f"{name:44s} {n:10,d} {nn:8d}")


def derive(f):
    f = f.copy()
    f["system"] = np.sqrt(f.aos_fwhm**2 + CAM_FWHM**2)  # idiq_aos_cam
    f["bud"] = budget(f.airmass)
    f["U"] = f.system / f.bud  # >1 == over budget
    f["U_aos"] = f.aos_fwhm / f.bud  # optics-only, for reference
    f["psf_fwhm"] = f.psf_sigma_median * SIGMA_TO_FWHM * PIXEL_SCALE
    return f


d = derive(d)
unguarded = derive(unguarded)
print(f"\nPRIMARY SAMPLE: {len(d):,} exposures on {d.day_obs.nunique()} nights")

## 5. Sample characterisation and selection bias

Two things to establish before any verdict: that the surviving sample spans enough airmass to test
the airmass clause, and that the guards have not selected a benign corner of parameter space.

In [ ]:
print("PRIMARY SAMPLE")
print(
    f"  exposures {len(d):,}   nights {d.day_obs.nunique()}   "
    f"day_obs {d.day_obs.min()} -> {d.day_obs.max()}"
)
for c, lbl in [
    ("airmass", "airmass"),
    ("dimm", "DIMM [arcsec]"),
    ("wind_speed", "wind [m/s]"),
    ("aos_fwhm", "aos_fwhm [arcsec]"),
    ("system", "system [arcsec]"),
]:
    q = d[c].quantile([0, 0.05, 0.5, 0.95, 1]).values
    print(
        f"  {lbl:20s} min={q[0]:6.3f} p5={q[1]:6.3f} med={q[2]:6.3f} p95={q[3]:6.3f} max={q[4]:6.3f}"
    )
print(
    f"  exposures with X > 1.4 : {(d.airmass > 1.4).sum():,}  "
    f"({100 * (d.airmass > 1.4).mean():.1f}%)   X > 2.0 : {(d.airmass > 2.0).sum():,}"
)
print(f"  bands: {d.band.value_counts().to_dict()}")

print("\nSELECTION-BIAS DIAGNOSTIC  (guarded vs dropped-by-spec-guards)")
dropped = unguarded.loc[~unguarded.index.isin(d.index)]
print(f"  {'quantity':24s} {'guarded':>10s} {'dropped':>10s}")
for c in ["airmass", "aos_fwhm", "system", "psf_fwhm", "dimm", "wind_speed"]:
    print(f"  {c:24s} {d[c].median():10.4f} {dropped[c].median():10.4f}")
print(
    "  (airmass/aos_fwhm similar => guards select on ATMOSPHERE, not on optics or pointing)"
)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 8.5))

a = ax[0, 0]
a.hist(
    unguarded.dimm, bins=np.linspace(0.2, 3, 60), color="lightgray", label="DIMM valid"
)
a.hist(d.dimm, bins=np.linspace(0.2, 3, 60), color="tab:blue", label="guarded")
a.axvline(
    SEEING_MED_RAW, color="crimson", lw=2, label=f'median seeing {SEEING_MED_RAW}"'
)
a.set_xlabel('DIMM seeing ["]')
a.set_ylabel("exposures")
a.legend(fontsize=8)
a.set_title("seeing guard")

a = ax[0, 1]
a.hist(unguarded.wind_speed.clip(0, 20), bins=40, color="lightgray", label="all")
a.hist(d.wind_speed.clip(0, 20), bins=40, color="tab:blue", label="guarded")
a.axvline(WIND_FLOOR, color="crimson", lw=2, label=f"DIMM floor {WIND_FLOOR} m/s")
a.set_xlabel("wind speed [m/s]")
a.legend(fontsize=8)
a.set_title("DIMM-reliability guard")

a = ax[0, 2]
a.hist(d.airmass, bins=np.linspace(1, 2.6, 50), color="tab:blue")
for X in (1.4, 2.0):
    a.axvline(X, color="crimson", ls="--", lw=1.5)
    a.text(X, a.get_ylim()[1] * 0.9, f" X={X}\n spec pt", color="crimson", fontsize=8)
a.set_xlabel("airmass")
a.set_title("airmass coverage of primary sample")

a = ax[1, 0]
a.hist(
    d.aos_fwhm,
    bins=np.linspace(0, 1.2, 60),
    color="tab:orange",
    label="aos_fwhm (optics)",
)
a.hist(
    d.system,
    bins=np.linspace(0, 1.2, 60),
    histtype="step",
    lw=2,
    color="k",
    label="system = aos (+) cam",
)
a.axvline(
    budget(1.0), color="crimson", lw=2, label=f'budget @zenith {budget(1.0):.3f}"'
)
a.set_xlabel('FWHM ["]')
a.set_ylabel("exposures")
a.legend(fontsize=8)
a.set_title("system term vs zenith budget")

a = ax[1, 1]
qs = pd.qcut(chk.dimm, 12)
gg = chk.groupby(qs, observed=True).agg(
    x=("dimm", "median"),
    y=("aos_fwhm", "median"),
    p90=("aos_fwhm", lambda s: s.quantile(0.9)),
)
a.plot(gg.x, gg.y, "o-", label="median aos_fwhm")
a.plot(gg.x, gg.p90, "s--", color="gray", label="p90")
a.axvline(SEEING_MED_RAW, color="crimson", lw=2, label="guard")
a.set_xlabel('DIMM seeing ["]')
a.set_ylabel('aos_fwhm ["]')
a.legend(fontsize=8)
a.set_title("masking check: aos_fwhm creeps up in bad seeing")

a = ax[1, 2]
a.hist(d.U, bins=np.linspace(0, 3, 60), color="tab:blue")
a.axvline(1.0, color="crimson", lw=2, label="budget")
a.axvline(d.U.median(), color="k", ls="--", lw=2, label=f"median {d.U.median():.2f}")
a.set_xlabel("U = system / budget")
a.legend(fontsize=8)
a.set_title("utilisation")

fig.suptitle("Guards and sample characterisation", fontsize=13)
fig.tight_layout()
plt.show()

## 6. Primary verdict

Utilisation $U = \mathrm{system}(X)\,/\,\mathrm{budget}(X)$, evaluated **per exposure at its own
airmass**. $U \le 1$ complies; $U > 1$ exceeds.

Per the repo convention ([feedback_within_night_fixed_effects](../dome_and_wind/Dome_Seeing_Model_Construction.ipynb)),
the effective sample size is **nights**, not exposures — exposures within a night share optical
state, thermal state and collimation history. The headline statistic is therefore the mean of
**night-median** $U$ with a standard error over nights.

In [ ]:
print("=" * 74)
print("PRIMARY VERDICT — U = system / budget, per exposure at its own airmass")
print("=" * 74)
print(f"sample: {len(d):,} exposures on {d.day_obs.nunique()} nights\n")

for lbl, col in [("aos_fwhm only (optics)", "U_aos"), ("system = aos (+) camera", "U")]:
    u = d[col]
    print(
        f"  {lbl:26s} median U={u.median():.3f}  p90={u.quantile(.9):.3f}  "
        f"frac>1={100 * (u > 1).mean():5.1f}%"
    )

nb = d.groupby("day_obs").agg(
    system=("system", "median"),
    aos=("aos_fwhm", "median"),
    bud=("bud", "median"),
    U=("U", "median"),
    n=("U", "size"),
)
m, se = nb.U.mean(), nb.U.std(ddof=1) / np.sqrt(len(nb))
t = (m - 1) / se
print(
    f"\n  night-median U : median {nb.U.median():.3f}   "
    f"IQR ({nb.U.quantile(.25):.3f}, {nb.U.quantile(.75):.3f})"
)
print(f"  mean of night-medians = {m:.3f} +/- {se:.3f} (SE over {len(nb)} nights)")
print(f"  95% CI = ({m - 1.96 * se:.3f}, {m + 1.96 * se:.3f})     t vs U=1 : {t:+.2f}")
print(
    f"  nights whose median exceeds budget : {(nb.U > 1).sum()} / {len(nb)} "
    f"({100 * (nb.U > 1).mean():.0f}%)"
)

lo = m - 1.96 * se
print(
    "\n  "
    + (
        "=> the 95% CI lies ENTIRELY ABOVE 1: exceedance is not a sampling fluctuation"
        if lo > 1
        else "=> CI includes 1: cannot reject compliance"
    )
)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

a = ax[0]
Xg = np.linspace(1, 2.6, 200)
a.plot(Xg, budget(Xg), "crimson", lw=2.5, label=r"budget $0.52(X/2)^{0.6}$")
a.scatter(
    d.airmass,
    d.system,
    s=3,
    alpha=0.12,
    color="tab:blue",
    label="system (per exposure)",
)
bins = [1.0, 1.1, 1.2, 1.35, 1.5, 1.75, 2.0, 3.5]
d["Xb"] = pd.cut(d.airmass, bins)
bt = d.groupby("Xb", observed=True).agg(
    X=("airmass", "median"),
    sysmed=("system", "median"),
    p90=("system", lambda s: s.quantile(0.9)),
    n=("system", "size"),
)
a.plot(bt.X, bt.sysmed, "ko-", lw=2, ms=7, label="binned median")
a.plot(bt.X, bt.p90, "k^--", lw=1.5, ms=6, label="binned p90")
for X, b in SPEC_POINTS.items():
    a.plot(X, b, "*", color="crimson", ms=18, zorder=5)
a.text(2.0, 0.545, " quoted\n spec pts", color="crimson", fontsize=8)
a.set_xlabel("airmass X")
a.set_ylabel('system FWHM ["]')
a.set_ylim(0, 1.0)
a.legend(fontsize=8, loc="upper left")
a.set_title("system term vs budget line")

a = ax[1]
a.plot(
    bt.X,
    100 * d.groupby("Xb", observed=True).U.apply(lambda s: (s > 1).mean()).values,
    "o-",
    lw=2,
    color="tab:red",
)
a.axhline(50, color="gray", ls=":", lw=1)
a.set_xlabel("airmass X")
a.set_ylabel("% exposures over budget")
a.set_title("exceedance is WORST AT ZENITH")
for x, y, n in zip(
    bt.X,
    100 * d.groupby("Xb", observed=True).U.apply(lambda s: (s > 1).mean()).values,
    bt.n,
):
    a.annotate(
        f"n={n}",
        (x, y),
        textcoords="offset points",
        xytext=(0, -14),
        fontsize=7,
        ha="center",
    )

a = ax[2]
a.plot(nb.index.astype(str), nb.U, "o", ms=4, color="tab:blue")
a.axhline(1.0, color="crimson", lw=2, label="budget")
a.axhline(m, color="k", ls="--", label=f"mean {m:.2f}")
a.set_xticks(a.get_xticks()[:: max(1, len(nb) // 8)])
a.tick_params(axis="x", rotation=60, labelsize=7)
a.set_ylabel("night-median U")
a.legend(fontsize=8)
a.set_title("per-night utilisation")

fig.tight_layout()
plt.show()

## 7. The airmass clause — is there a gravity-load signature?

The spec calls out *"seeing degradation due to hardware away from the zenith (e.g., due to gravity
load)"*. The budget it allows grows as $X^{0.6}$. So the clause is really a statement about the
**slope** of the system term against airmass:

- If the system term grew as $X^{0.6}$ too, utilisation would be flat and the clause would be
  satisfied at all $X$ if satisfied at one.
- If it is **flat** in $X$, there is *no* detectable gravity-load degradation, and utilisation is
  worst at **zenith** — the opposite of the concern the clause was written to guard against.

Tested within-night (night-demeaned, cluster-robust SEs on night) with a within-night placebo and a
positive control, per the repo's required checks.

In [ ]:
def cluster_ols(x, y, groups):
    """OLS slope with cluster-robust SE on `groups`."""
    x, y, groups = np.asarray(x, float), np.asarray(y, float), np.asarray(groups)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, groups = x[ok], y[ok], groups[ok]
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta
    XtX_inv = np.linalg.inv(X.T @ X)
    meat = np.zeros((2, 2))
    for g in np.unique(groups):
        i = groups == g
        Xg, rg = X[i], resid[i]
        meat += Xg.T @ np.outer(rg, rg) @ Xg
    V = XtX_inv @ meat @ XtX_inv
    ng = len(np.unique(groups))
    V *= ng / max(ng - 1, 1)
    return beta[1], float(np.sqrt(V[1, 1]))


def add_within(f, cols, by="day_obs"):
    f = f.copy()
    for c in cols:
        f[c + "_w"] = f[c] - f.groupby(by)[c].transform("median")
    return f


w = d.copy()
w["lX"] = np.log(w.airmass)
w["lA"] = np.log(w.aos_fwhm)
w["lS"] = np.log(w.system)
w["lP"] = np.log(w.psf_fwhm)
w = add_within(w, ["lX", "lA", "lS", "lP"])

print("d ln(term) / d ln(X)   -- spec budget slope is +0.60")
print("-" * 70)
for lbl, col in [("aos_fwhm (optics)", "lA"), ("system (aos+cam)", "lS")]:
    sp, sp_e = cluster_ols(w.lX, w[col], w.day_obs)
    sw, sw_e = cluster_ols(w.lX_w, w[col + "_w"], w.day_obs)
    print(
        f"{lbl:20s} pooled {sp:+.4f} +/- {sp_e:.4f}    within-night {sw:+.4f} +/- {sw_e:.4f}"
    )
    print(
        f"{'':20s}   within: t vs 0 = {sw / sw_e:+.2f}   t vs +0.60 = {(sw - ALPHA) / sw_e:+.2f}"
    )

slope_aos, slope_aos_se = cluster_ols(w.lX_w, w.lA_w, w.day_obs)

print("\nPOSITIVE CONTROL — delivered PSF vs airmass (atmosphere must show ~+0.6)")
sc, sc_e = cluster_ols(w.lX_w, w.lP_w, w.day_obs)
print(
    f"  d ln(PSF)/d ln(X) within-night = {sc:+.4f} +/- {sc_e:.4f}   -> t vs 0 = {sc / sc_e:+.2f}"
)
print(
    "  (recovers the atmospheric airmass response => the design has power to detect a +0.6 slope)"
)

print("\nPLACEBO — permute airmass within night (200 draws)")
ts = []
for _ in range(200):
    perm = w.groupby("day_obs", group_keys=False).lX_w.transform(
        lambda s: RNG.permutation(s.values)
    )
    ts.append(np.polyfit(perm, w.lA_w, 1)[0])
ts = np.array(ts)
print(
    f"  real slope {slope_aos:+.4f}   placebo mean {ts.mean():+.5f} sd {ts.std():.5f}"
)
print(
    f"  placebo 95% range ({np.percentile(ts, 2.5):+.4f}, {np.percentile(ts, 97.5):+.4f})"
)

print("\nBINNED — system, budget and utilisation vs airmass")
bt2 = d.groupby("Xb", observed=True).agg(
    n=("U", "size"),
    X=("airmass", "median"),
    aos=("aos_fwhm", "median"),
    system=("system", "median"),
    budget=("bud", "median"),
    U=("U", "median"),
    pct_over=("U", lambda s: 100 * (s > 1).mean()),
)
print(bt2.round(3).to_string())
print(
    f"\n=> system term is essentially FLAT in X (slope {slope_aos:+.3f}, excluded from +0.60 at "
    f"{abs(slope_aos - ALPHA) / slope_aos_se:.1f} sigma):"
)
print(
    "   NO gravity-load degradation is detected. Utilisation therefore falls with airmass and the"
)
print(
    "   binding case is ZENITH, not X=2 -- the airmass clause is satisfied in its own terms."
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))

a = ax[0]
Xg = np.linspace(1, 2.3, 100)
a.plot(
    Xg, budget(Xg) / budget(1.0), "crimson", lw=2.5, label=r"budget $\propto X^{0.6}$"
)
a.plot(
    Xg,
    (Xg**slope_aos),
    "tab:blue",
    lw=2.5,
    label=f"measured optics $\\propto X^{{{slope_aos:+.2f}}}$",
)
a.fill_between(
    Xg,
    Xg ** (slope_aos - 1.96 * slope_aos_se),
    Xg ** (slope_aos + 1.96 * slope_aos_se),
    color="tab:blue",
    alpha=0.2,
    label="95% CI",
)
a.set_xlabel("airmass X")
a.set_ylabel("relative to zenith")
a.legend(fontsize=9)
a.set_title("budget grows with X; the system term does not")

a = ax[1]
a.plot(bt2.X, bt2.U, "o-", lw=2, color="tab:blue", label="median U")
a.axhline(1.0, color="crimson", lw=2, label="budget")
a.set_xlabel("airmass X")
a.set_ylabel("U = system / budget")
a.legend(fontsize=9)
a.set_title("utilisation is worst at zenith")
fig.tight_layout()
plt.show()

## 8. Guard sensitivity

A verdict that depends on our cuts is not a verdict. Re-run against the budget line while relaxing
each guard, and against the two alternative budget readings we rejected in §2.

In [ ]:
print("GUARD SENSITIVITY — median U and exceedance under each variant")
print("-" * 88)
variants = [
    ("primary (DIMM<=0.69, wind>=4)", d),
    ("drop wind floor", unguarded[unguarded.dimm <= SEEING_MED_RAW]),
    ("drop seeing guard", unguarded[unguarded.wind_speed >= WIND_FLOOR]),
    ("drop both (all conditions)", unguarded),
    (
        'near-median band 0.60-0.80"',
        unguarded[
            unguarded.dimm.between(0.60, 0.80) & (unguarded.wind_speed >= WIND_FLOOR)
        ],
    ),
    (
        'stricter seeing <=0.604"',
        unguarded[
            (unguarded.dimm <= THETA0_CORR) & (unguarded.wind_speed >= WIND_FLOOR)
        ],
    ),
    ("X>1.4 only", d[d.airmass > 1.4]),
]
print(
    f"{'variant':32s} {'n':>7s} {'nights':>7s} {'medU':>7s} {'%>1':>6s} {'meanU':>7s} {'t vs 1':>7s}"
)
for name, s in variants:
    s = derive(s) if "U" not in s.columns else s
    nbv = s.groupby("day_obs").U.median()
    mv = nbv.mean()
    sev = nbv.std(ddof=1) / np.sqrt(len(nbv))
    print(
        f"{name:32s} {len(s):7,d} {s.day_obs.nunique():7d} {s.U.median():7.3f} "
        f"{100 * (s.U > 1).mean():6.1f} {mv:7.3f} {(mv - 1) / sev:+7.2f}"
    )

print("\nBUDGET-DEFINITION SENSITIVITY (primary sample)")
print("-" * 88)
# each reading expressed as a function of X, so the zenith value is exact rather than inferred
readings = [
    ("adopted: 0.52*(X/2)^0.6", budget),
    (
        "raw-0.69 base (lenient)",
        lambda X: SEEING_MED_RAW * np.asarray(X, float) ** ALPHA * FRAC_QUAD,
    ),
    ('flat 0.52" at all X', lambda X: np.full_like(np.asarray(X, float), 0.52)),
    (
        "variance reading sqrt(0.15)",
        lambda X: SEEING_MED_RAW * np.asarray(X, float) ** ALPHA * FRAC_VAR,
    ),
]
print(f"{'reading':32s} {'@zenith':>8s} {'@X=1.4':>8s} {'medU':>8s} {'%>1':>6s}")
for name, fn in readings:
    u = d.system / fn(d.airmass)
    print(
        f"{name:32s} {float(fn(1.0)):8.4f} {float(fn(1.4)):8.4f} "
        f"{u.median():8.3f} {100 * (u > 1).mean():6.1f}"
    )

print("\nOptics-only (excluding the camera floor), primary sample:")
print(f"  median U_aos {d.U_aos.median():.3f}   %>1 {100 * (d.U_aos > 1).mean():.1f}")
print(
    "\n=> the exceedance survives every guard relaxation and is NOT an artifact of the cuts."
)
print(
    "   It is defeated only by the flat-0.52\" reading, which the spec's own two points exclude."
)

## 9. Time trend — commissioning maturation

`aos_fwhm` is a *commissioning-era* measurement of a system still being collimated and tuned. A
static verdict over 15 months would conflate early alignment with current performance, so we check
whether the exceedance is shrinking.

In [ ]:
d["month"] = d.day_obs // 100
mo = d.groupby("month").agg(
    n=("U", "size"),
    nights=("day_obs", "nunique"),
    aos=("aos_fwhm", "median"),
    U=("U", "median"),
)
print("monthly medians")
print(mo.round(3).to_string())

nbt = d.groupby("day_obs").agg(U=("U", "median"), aos=("aos_fwhm", "median"))
rho, pval = spearmanr(nbt.index.values, nbt.U.values)
print(
    f"\nSpearman(day_obs, night-median U) = {rho:+.4f}   p = {pval:.2e}   ({len(nbt)} nights)"
)

recent = d[d.day_obs >= 20260401]
rnb = recent.groupby("day_obs").U.median()
mr, ser = rnb.mean(), rnb.std(ddof=1) / np.sqrt(len(rnb))
print(
    f"\nMOST RECENT EPOCH (day_obs >= 20260401): {len(recent):,} exposures, {len(rnb)} nights"
)
print(
    f"  median U = {recent.U.median():.3f}   mean night-median U = {mr:.3f} +/- {ser:.3f}"
    f"   t vs 1 = {(mr - 1) / ser:+.2f}"
)
print(
    f'  %>1 = {100 * (recent.U > 1).mean():.1f}   median aos_fwhm = {recent.aos_fwhm.median():.4f}"'
)
early = d[d.day_obs < 20260101]
print(
    f'  vs pre-2026: median U {early.U.median():.3f}, median aos_fwhm {early.aos_fwhm.median():.4f}"'
)
print(
    f"\n=> optics ARE improving ({100 * (1 - recent.aos_fwhm.median() / early.aos_fwhm.median()):.0f}% "
    f"reduction in aos_fwhm), but the most recent epoch is still above budget."
)

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))
a = ax[0]
a.plot(range(len(nbt)), nbt.U, "o", ms=4, alpha=0.7)
k = max(3, len(nbt) // 12)
a.plot(
    range(len(nbt)),
    nbt.U.rolling(k, center=True, min_periods=1).median(),
    "k-",
    lw=2,
    label=f"rolling median ({k} nights)",
)
a.axhline(1.0, color="crimson", lw=2, label="budget")
a.set_xlabel("night index (chronological)")
a.set_ylabel("night-median U")
a.legend(fontsize=8)
a.set_title(f"maturation: Spearman rho={rho:+.2f}, p={pval:.1e}")

a = ax[1]
a.errorbar(range(len(mo)), mo.U, fmt="o-", lw=2)
a.axhline(1.0, color="crimson", lw=2)
a.set_xticks(range(len(mo)))
a.set_xticklabels(mo.index.astype(str), rotation=60, fontsize=7)
a.set_ylabel("median U")
a.set_title("monthly median utilisation")
fig.tight_layout()
plt.show()

## 10. Where the budget actually goes

At zenith the strict budget is $0.343''$, and the camera/detector floor alone is $0.207''$. Since the
terms add in quadrature, the floor consumes a large fraction of the budget *before any optics
contribute at all* — worth quantifying, because it determines whether the exceedance is an optics
problem or a budget-definition problem.

In [ ]:
b1 = budget(1.0)
allowed_optics = np.sqrt(max(b1**2 - CAM_FWHM**2, 0))
print(f'At zenith (X=1), strict budget = {b1:.4f}"')
print(
    f'  camera floor           {CAM_FWHM:.4f}"  = {100 * CAM_FWHM / b1:.1f}% of the budget in FWHM'
)
print(
    f"                                    = {100 * (CAM_FWHM / b1) ** 2:.1f}% of the budget in QUADRATURE"
)
print(
    f'  => allowed optics term {allowed_optics:.4f}"   (what aos_fwhm would have to fit under)'
)
print(
    f'  observed median aos_fwhm {d.aos_fwhm.median():.4f}"  '
    f"-> needs a {d.aos_fwhm.median() / allowed_optics:.2f}x reduction"
)

print(f"\nsame at the two quoted spec points:")
for X, b in sorted(SPEC_POINTS.items()):
    ao = np.sqrt(max(b**2 - CAM_FWHM**2, 0))
    print(
        f'  X={X}: budget {b:.3f}"  camera {CAM_FWHM}"  -> allowed optics {ao:.4f}"  '
        f"({100 * (CAM_FWHM / b) ** 2:.0f}% of variance spent on the detector)"
    )

print("\nvariance decomposition of the observed system term (primary sample medians):")
v_cam = CAM_FWHM**2
v_opt = d.aos_fwhm.median() ** 2
print(f"  optics  {v_opt:.5f} arcsec^2  ({100 * v_opt / (v_opt + v_cam):.1f}%)")
print(f"  camera  {v_cam:.5f} arcsec^2  ({100 * v_cam / (v_opt + v_cam):.1f}%)")
print(f'  system  {np.sqrt(v_opt + v_cam):.4f}"  vs zenith budget {b1:.4f}"')
print(
    "\n=> the camera floor is a fixed, irreducible 36% of the zenith variance budget. Even"
)
print(
    '   perfect optics would leave only ~0.27" of headroom, so the zenith case is intrinsically'
)
print(
    "   tight; conversely, the observed exceedance cannot be blamed on the camera alone, since"
)
print(
    "   aos_fwhm ALONE already exceeds the zenith budget (U_aos median "
    f"{d.U_aos.median():.2f})."
)

## 11. Verdict

Using the taxonomy from the wind notebook, extended with the distinction that matters here.
`U = 95%-confidence-bounded utilisation at the relevant airmass`.

In [ ]:
def verdict(mean_u, se_u, has_absolute_budget=True):
    lo, hi = mean_u - 1.96 * se_u, mean_u + 1.96 * se_u
    t = (mean_u - 1) / se_u
    if hi <= 1:
        return "CONSISTENT", lo, hi, t
    if lo > 1:
        return ("CONTRADICTED" if has_absolute_budget else "NO-BUDGET"), lo, hi, t
    return "UNDERPOWERED", lo, hi, t


rows = []
for lbl, s in [
    ("full campaign (85 nights)", d),
    ("recent epoch (>=20260401)", d[d.day_obs >= 20260401]),
    ("X > 1.4 subset", d[d.airmass > 1.4]),
    ("optics only, full campaign", d),
]:
    col = "U_aos" if lbl.startswith("optics") else "U"
    nbv = s.groupby("day_obs")[col].median()
    mv, sev = nbv.mean(), nbv.std(ddof=1) / np.sqrt(len(nbv))
    v, lo, hi, t = verdict(mv, sev)
    rows.append(
        {
            "sample": lbl,
            "nights": len(nbv),
            "mean_U": round(mv, 3),
            "CI_lo": round(lo, 3),
            "CI_hi": round(hi, 3),
            "t_vs_1": round(t, 2),
            "verdict": v,
        }
    )
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

print("\n" + "=" * 78)
print("AIRMASS CLAUSE")
print("=" * 78)
print(f"  within-night d ln(aos)/d ln(X) = {slope_aos:+.4f} +/- {slope_aos_se:.4f}")
print(
    f"  excluded from the budget slope +0.60 at {abs(slope_aos - ALPHA) / slope_aos_se:.1f} sigma"
)
print(
    "  verdict: NO gravity-load degradation detected -> the airmass clause is SATISFIED,"
)
print("           and the binding airmass is ZENITH (where the budget is smallest).")

### Findings

**1. The spec decodes exactly, and self-consistently.** The two quoted numbers are redundant, and
both return the same implied zenith seeing, $0.604''$, under the quadrature reading of "15%"
($\mathrm{sys} = \theta_{\rm atm}\sqrt{1.15^2-1}$). That value is the outer-scale-corrected form of
the site's raw $0.69''$ median ($0.69/0.604 = 1.142$). The budget line
$0.52(X/2)^{0.6}$ reproduces both quoted points to <2 mas.

**2. The system term exceeds the budget — CONTRADICTED.** On 85 guarded nights the mean night-median
utilisation is $U = 1.316 \pm 0.041$, a 95% CI of $(1.237, 1.396)$ entirely above 1, $t = +7.8$;
80 of 85 nights exceed. Unlike the wind spec, this design point *is* reached on sky, so this is a
genuine contradiction rather than a power limitation. It survives every guard relaxation.

**3. No gravity-load signature; the binding case is zenith.** The optics term is flat in airmass
(within-night $d\ln(\mathrm{aos})/d\ln X = +0.089 \pm 0.053$, excluded from $+0.60$ at $9.7\sigma$),
with a clean placebo ($-0.013$) and a working positive control (PSF recovers $+0.41$). So the
specific worry the airmass clause encodes — hardware degrading away from zenith — is **not
observed**. Because the budget grows as $X^{0.6}$ while the hardware does not, exceedance is worst at
zenith (97% of exposures over budget at $X<1.1$, 25% at $X>2$). The spec is hardest to meet exactly
where the clause was least concerned.

**4. Both the optics and the camera floor matter.** `aos_fwhm` alone already exceeds the zenith
budget (median $U_{\rm aos} = 1.07$, mean night-median $1.19$ with CI $(1.11, 1.27)$), so this is not an artifact of
adding the camera term. But the $0.207''$ camera floor is **36% of the zenith variance budget**
before any optics contribute, leaving only $0.274''$ of headroom for the optics — the observed
median `aos_fwhm` of $0.399''$ needs a $1.46\times$ reduction.

**5. The system is measurably improving.** Median `aos_fwhm` fell from $0.442''$ (pre-2026) to
$0.382''$ (since 2026-04-01), a 13% reduction; Spearman $\rho = -0.49$ ($p = 2\times10^{-6}$) against
night. The most recent epoch sits at $U = 1.17 \pm 0.03$ — still above budget, but far closer than
the campaign average. The
trend is the most encouraging result here and argues for re-running this notebook as data accumulate.

### Caveats

- **`aos_fwhm` is a model-derived quantity**, not a direct measurement: it is a Zernike-based
  estimate whose accuracy in the FWHM domain is not independently validated here. §4 shows it drifts
  upward in poor seeing (pooled $r=+0.089$, a $+4.9\%$ rise in median `aos_fwhm` across the seeing range), which is why the median-seeing guard is applied. A
  systematic bias in the Zernike→FWHM conversion would move the verdict; that conversion should be
  checked against an independent instrumental-PSF estimate before treating this as final.
- **Camera-floor provenance.** $0.207''$ comes from `rubin_nights`, not from the spec document. If
  the spec's "system" excludes the detector, the correct comparison is the optics-only row
  ($U_{\rm aos}$), which still exceeds budget but by less.
- **Commissioning era.** The window opens 2025-04-15, well inside commissioning; finding 5 suggests
  the campaign-average verdict understates current performance.
- **`seeing_zenith_500nm_median` is 100% null** in this window (as in the sibling notebooks), so the
  independent zenith-referenced seeing cross-check is unavailable.

### Recommended next step

Validate the Zernike→FWHM conversion. Every conclusion above rests on `aos_fwhm` being an unbiased
FWHM-equivalent of the wavefront residual, and that is the one assumption this notebook cannot test
from ConsDB alone. The natural check is against the `donut_blur_fwhm`/`psf` decomposition, or against
open-loop hexapod scans where the induced aberration is known — the same intervention-over-passive-data
argument that the dome-seeing notebook reached for $g(V)$.